# exp_2a — Experiment 2A: MLP for California Housing Regression

Source-based notebook converted from the uploaded experiment code document.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam, SGD

import keras_tuner as kt

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
path = r"D:\DL DATASET\housing.csv (1).zip"

df = pd.read_csv(path)

print(df.head())

In [ ]:
df["total_bedrooms"] = df["total_bedrooms"].fillna(
    df["total_bedrooms"].median()
)

In [ ]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
X_train = pd.get_dummies(
    X_train,
    columns=["ocean_proximity"],
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    columns=["ocean_proximity"],
    drop_first=True
)

# Ensure same columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [ ]:
import numpy as np

print("NaN in y_train:", np.isnan(y_train).sum())
print("Inf in y_train:", np.isinf(y_train).sum())
print("Min:", y_train.min())
print("Max:", y_train.max())

In [ ]:
print(X_train.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test = y_scaler.transform(y_test.values.reshape(-1, 1))

In [ ]:
def build_model(hp):

    model = Sequential()

    model.add(Input(shape=(X_train.shape[1],)))# First Hidden Layer
    model.add(
        Dense(
            units=hp.Int(
                "units",
                min_value=32,
                max_value=128,
                step=32
            ),
            activation="relu"
        )
    )

    # Dropout
    model.add(
        Dropout(
            hp.Float(
                "dropout",
                0.0,
                0.4,
                step=0.2
            )
        )
    )

    # Additional Hidden Layers
    for i in range(
        hp.Int("hidden_layers", 1, 2)
    ):

        model.add(
            Dense(
                units=hp.Int(
                    f"units_{i}",
                    min_value=32,
                    max_value=128,
                    step=32
                ),
                activation="relu"
            )
        )    # Output Layer
    model.add(Dense(1))

    # Optimizer
    optimizer = hp.Choice(
        "optimizer",
        ["adam", "sgd"]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [0.01, 0.001, 0.0001] # Added 0.0001 for more stability
    )

    if optimizer == "adam":
        opt = Adam(learning_rate=learning_rate)
    else:
        opt = SGD(learning_rate=learning_rate)

    model.compile(
                optimizer=opt,
        loss="mse",
        metrics=["mae"]
    )

    return model

# ==========================================================
# Hyperparameter Tuning
# ==========================================================

tuner = kt.RandomSearch(
    build_model,
    objective="val_loss",
    max_trials=5,
    overwrite=True,
    directory="Regression_Tuning",
    project_name="CaliforniaHousing"
)

In [ ]:
tuner.search(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=10,
    batch_size=32,
    verbose=1
)

In [ ]:
best_model = tuner.get_best_models(1)[0]

print("\nBest Hyperparameters\n")

best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)

In [ ]:
history = best_model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=15,
    batch_size=32,
    verbose=1
)

In [ ]:
loss, mae = best_model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("\nTest MSE :", loss)
print("Test MAE :", mae)

In [ ]:
predictions = best_model.predict(X_test)

predictions_original = y_scaler.inverse_transform(predictions)
actual_original = y_scaler.inverse_transform(y_test)

comparison = pd.DataFrame({
    "Actual": actual_original.flatten()[:10],
    "Predicted": predictions_original.flatten()[:10]
})

print(comparison)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("Training vs Validation Loss")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history["mae"],
    label="Training MAE"
)

plt.plot(
    history.history["val_mae"],
    label="Validation MAE"
)
plt.xlabel("Epoch")
plt.ylabel("Mean Absolute Error")

plt.title("Training vs Validation MAE")

plt.legend()

plt.show()